In [4]:
!pip install mediapipe opencv-python scikit-learn numpy matplotlib tqdm fastapi uvicorn pydantic torch --quiet
print('\n✅ All packages installed!')


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

✅ All packages installed!


In [6]:
import os, cv2, time, numpy as np

# ══════════════════════════════════════════════════════════════════
# 21 ISL word-glosses
WORD_GLOSSES = [
    'GOOD', 'MORNING', 'NIGHT', 'AFTERNOON',
    'HOW', 'YOU', 'FINE', 'THANK',
    'PLEASE', 'HELP', 'SORRY', 'WHAT',
    'DRINK', 'WATER', 'EAT', 'HOME',
    'NICE', 'NAME', 'MEET', 'WHO', 'WANT','NAMASTE'
]

# 26 ISL fingerspelling alphabets (A–Z)
# Recorded exactly like word glosses — hold the handshape for ~2 seconds
ALPHA_GLOSSES = [chr(i) for i in range(ord('A'), ord('Z') + 1)]   # ['A','B',...,'Z']

# Combined list — model will predict any of these 47 classes
GLOSSES = WORD_GLOSSES + ALPHA_GLOSSES

SAMPLES_PER_CLASS = 30    # video clips per class
FRAMES_PER_SAMPLE = 60    # frames per clip (~2 sec at 30 fps)
DATASET_DIR       = 'csl_dataset'
CAM_INDEX         = 0

print(f'Total classes  : {len(GLOSSES)}')
print(f'  Word glosses : {len(WORD_GLOSSES)}  → {WORD_GLOSSES}')
print(f'  Alphabets    : {len(ALPHA_GLOSSES)}  → {ALPHA_GLOSSES}')
print(f'Samples/class  : {SAMPLES_PER_CLASS}')
print(f'Total clips    : {len(GLOSSES) * SAMPLES_PER_CLASS}')
print()
print('TIP — For alphabets: hold your hand still in the ISL handshape')
print('      for the full 60 frames. Consistent position = better accuracy.')

Total classes  : 48
  Word glosses : 22  → ['GOOD', 'MORNING', 'NIGHT', 'AFTERNOON', 'HOW', 'YOU', 'FINE', 'THANK', 'PLEASE', 'HELP', 'SORRY', 'WHAT', 'DRINK', 'WATER', 'EAT', 'HOME', 'NICE', 'NAME', 'MEET', 'WHO', 'WANT', 'NAMASTE']
  Alphabets    : 26  → ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']
Samples/class  : 30
Total clips    : 1440

TIP — For alphabets: hold your hand still in the ISL handshape
      for the full 60 frames. Consistent position = better accuracy.


In [7]:
# Create folders
os.makedirs(DATASET_DIR, exist_ok=True)
for g in GLOSSES:
    os.makedirs(os.path.join(DATASET_DIR, g), exist_ok=True)

cap = cv2.VideoCapture(CAM_INDEX)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

def overlay(frame, lines, y=50, color=(255,255,255), scale=0.7, thick=2):
    for i, ln in enumerate(lines):
        cv2.putText(frame, ln, (20, y + i*35),
                    cv2.FONT_HERSHEY_SIMPLEX, scale, color, thick, cv2.LINE_AA)

def count_done(gloss):
    d = os.path.join(DATASET_DIR, gloss)
    return len([x for x in os.listdir(d) if os.path.isdir(os.path.join(d, x))])

# ── Hint: what ISL handshape to make for each letter ──────────────────────
ALPHA_HINT = {
    'A':'Closed fist, thumb side',   'B':'Flat open hand, fingers up',
    'C':'Curved C shape',            'D':'Index up, others curve to thumb',
    'E':'All fingers hooked',        'F':'Index-thumb circle, 3 fingers up',
    'G':'Index+thumb point sideways','H':'2 fingers extended sideways',
    'I':'Pinky only raised',         'J':'Pinky up, trace J in air',
    'K':'Index up, middle bent, thumb between',
    'L':'L shape — thumb+index',     'M':'3 fingers over tucked thumb',
    'N':'2 fingers over tucked thumb','O':'All fingertips meet in O',
    'P':'Like K but pointing down',  'Q':'Like G but hanging down',
    'R':'Index+middle crossed',      'S':'Tight fist, thumb over',
    'T':'Thumb between index+middle','U':'2 fingers together pointing up',
    'V':'V/peace sign',              'W':'3 fingers spread, pinky+thumb',
    'X':'Index hooked/bent',         'Y':'Thumb+pinky out',
    'Z':'Index traces Z'
}

gloss_idx = 0
print('Press S=record  N=skip  Q=quit')

while gloss_idx < len(GLOSSES):
    gloss = GLOSSES[gloss_idx]
    done  = count_done(gloss)
    rem   = SAMPLES_PER_CLASS - done

    if rem <= 0:
        print(f'  ✅ {gloss} already complete — skipping')
        gloss_idx += 1
        continue

    is_alpha = gloss in ALPHA_GLOSSES
    hint = ALPHA_HINT.get(gloss, '') if is_alpha else ''

    # ── Waiting state ──────────────────────────────────────────────────────
    while True:
        ret, frame = cap.read()
        if not ret: break
        frame = cv2.flip(frame, 1)
        label_type = f'[ALPHABET]' if is_alpha else '[GLOSS]'
        lines = [
            f'{label_type}  {gloss}  ({gloss_idx+1}/{len(GLOSSES)})',
            f'Collected: {done}/{SAMPLES_PER_CLASS}',
        ]
        if hint:
            lines.append(f'Handshape: {hint}')
        lines += ['', 'S = record   N = skip   Q = quit']
        overlay(frame, lines, color=(50,220,50) if not is_alpha else (50,180,255))
        cv2.imshow('ISL CSL Collector', frame)
        key = cv2.waitKey(1) & 0xFF
        if key == ord('s'): break
        elif key == ord('n'): gloss_idx += 1; break
        elif key == ord('q'): gloss_idx = len(GLOSSES); break

    if gloss_idx >= len(GLOSSES): break

    # ── Countdown 3-2-1 ───────────────────────────────────────────────────
    for cnt in [3, 2, 1]:
        deadline = time.time() + 1.0
        while time.time() < deadline:
            ret, frame = cap.read()
            if not ret: break
            frame = cv2.flip(frame, 1)
            overlay(frame, [f'Starting in {cnt}...'], y=220,
                    color=(0,200,255), scale=1.4, thick=3)
            cv2.imshow('ISL CSL Collector', frame)
            cv2.waitKey(1)

    # ── Record frames ─────────────────────────────────────────────────────
    sample_dir = os.path.join(DATASET_DIR, gloss, f'sample_{done:03d}')
    os.makedirs(sample_dir, exist_ok=True)
    for fi in range(FRAMES_PER_SAMPLE):
        ret, frame = cap.read()
        if not ret: break
        frame = cv2.flip(frame, 1)
        cv2.imwrite(os.path.join(sample_dir, f'frame_{fi:03d}.jpg'), frame)
        prog = int((fi / FRAMES_PER_SAMPLE) * 200)
        cv2.rectangle(frame, (20,440), (20+prog,460), (0,140,255), -1)
        cv2.rectangle(frame, (20,440), (220,460), (200,200,200), 2)
        overlay(frame, [f'RECORDING {gloss}', f'Frame {fi+1}/{FRAMES_PER_SAMPLE}'],
                color=(0,80,255), scale=0.9, thick=2)
        cv2.imshow('ISL CSL Collector', frame)
        cv2.waitKey(1)

    done += 1; rem -= 1
    print(f'  Saved: {gloss}/sample_{done-1:03d}  ({done}/{SAMPLES_PER_CLASS})')

    if rem <= 0:
        gloss_idx += 1
    else:
        # 1.5s pause between samples
        t_end = time.time() + 1.5
        while time.time() < t_end:
            ret, f = cap.read()
            if not ret: break
            f = cv2.flip(f, 1)
            overlay(f, ['✅ Saved!  Reset hand...'], y=220, color=(0,220,120), scale=0.85)
            cv2.imshow('ISL CSL Collector', f)
            cv2.waitKey(1)

cap.release()
cv2.destroyAllWindows()
print('\n✅ Data collection complete!')
for g in GLOSSES:
    n = count_done(g)
    tag = '✅' if n >= SAMPLES_PER_CLASS else f'⚠️ {n}/{SAMPLES_PER_CLASS}'
    print(f'  {g:12s}: {tag}')

Press S=record  N=skip  Q=quit
  Saved: GOOD/sample_000  (1/30)
  Saved: GOOD/sample_001  (2/30)
  Saved: GOOD/sample_002  (3/30)
  Saved: GOOD/sample_003  (4/30)
  Saved: GOOD/sample_004  (5/30)
  Saved: GOOD/sample_005  (6/30)
  Saved: GOOD/sample_006  (7/30)
  Saved: GOOD/sample_007  (8/30)
  Saved: GOOD/sample_008  (9/30)
  Saved: GOOD/sample_009  (10/30)
  Saved: GOOD/sample_010  (11/30)
  Saved: GOOD/sample_011  (12/30)
  Saved: GOOD/sample_012  (13/30)
  Saved: GOOD/sample_013  (14/30)
  Saved: GOOD/sample_014  (15/30)
  Saved: GOOD/sample_015  (16/30)
  Saved: MORNING/sample_000  (1/30)
  Saved: MORNING/sample_001  (2/30)
  Saved: MORNING/sample_002  (3/30)
  Saved: MORNING/sample_003  (4/30)
  Saved: MORNING/sample_004  (5/30)
  Saved: MORNING/sample_005  (6/30)
  Saved: MORNING/sample_006  (7/30)
  Saved: MORNING/sample_007  (8/30)
  Saved: MORNING/sample_008  (9/30)
  Saved: MORNING/sample_009  (10/30)
  Saved: MORNING/sample_010  (11/30)
  Saved: MORNING/sample_011  (12/30)

In [4]:
import os, urllib.request
import numpy as np
import cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from tqdm import tqdm

# ════════════════════════════════════════════
MAX_FRAMES  = 60           # must match FRAMES_PER_SAMPLE
DATASET_DIR = 'csl_dataset'
OUTPUT_PATH = 'data/csl_keypoints.npz'
IMG_EXTS    = ('.jpg', '.jpeg', '.png', '.bmp')

WORD_GLOSSES = [
    'GOOD', 'MORNING', 'NIGHT', 'AFTERNOON',
    'HOW', 'YOU', 'FINE', 'THANK',
    'PLEASE', 'HELP', 'SORRY', 'WHAT',
    'DRINK', 'WATER', 'EAT', 'HOME',
    'NICE', 'NAME', 'MEET', 'WHO', 'WANT'
]
ALPHA_GLOSSES = [chr(i) for i in range(ord('A'), ord('Z') + 1)]
GLOSSES       = WORD_GLOSSES + ALPHA_GLOSSES   # 47 total
# ════════════════════════════════════════════

os.makedirs('data', exist_ok=True)

# ── Download hand landmarker model ────────────────────────────────────────
MODEL_PATH = 'hand_landmarker.task'
if not os.path.exists(MODEL_PATH):
    print('Downloading hand landmarker model (~9MB)...')
    url = ('https://storage.googleapis.com/mediapipe-models/'
           'hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task')
    urllib.request.urlretrieve(url, MODEL_PATH)
    print('Downloaded:', MODEL_PATH)
else:
    print('Model file already exists:', MODEL_PATH)

# ── Build detector ────────────────────────────────────────────────────────
base_options = mp_python.BaseOptions(model_asset_path=MODEL_PATH)
options = mp_vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=1,
    min_hand_detection_confidence=0.4,
    min_hand_presence_confidence=0.4,
    min_tracking_confidence=0.4,
    running_mode=mp_vision.RunningMode.IMAGE,
)
detector = mp_vision.HandLandmarker.create_from_options(options)

# ── Helpers ───────────────────────────────────────────────────────────────
def normalize(kps):
    pts   = kps.reshape(21, 3)
    pts   = pts - pts[0]             # wrist to origin
    scale = np.linalg.norm(pts[9])   # middle-finger MCP distance
    if scale > 0:
        pts /= scale
    return pts.flatten()             # (63,)

def detect_frame(img_bgr):
    img_rgb  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
    result   = detector.detect(mp_image)
    if not result.hand_landmarks:
        return None
    lms = result.hand_landmarks[0]
    kps = np.array([[lm.x, lm.y, lm.z] for lm in lms], dtype=np.float32).flatten()
    return normalize(kps)

def pad_or_truncate(seq, max_frames):
    arr = np.array(seq, dtype=np.float32)   # (T, 63)
    if len(arr) >= max_frames:
        return arr[:max_frames]
    pad = np.zeros((max_frames - len(arr), 63), dtype=np.float32)
    return np.concatenate([arr, pad], axis=0)

# ── Main extraction loop ──────────────────────────────────────────────────
X, y = [], []
failed  = 0
missing = []

for gloss in GLOSSES:
    gloss_dir = os.path.join(DATASET_DIR, gloss)

    if not os.path.isdir(gloss_dir):
        missing.append(gloss)
        continue

    samples = sorted([s for s in os.listdir(gloss_dir)
                      if os.path.isdir(os.path.join(gloss_dir, s))])
    extracted = 0

    for sample in tqdm(samples, desc=f'{gloss} ({len(samples)} samples)', leave=False):
        sample_dir   = os.path.join(gloss_dir, sample)
        frames_sorted = sorted([f for f in os.listdir(sample_dir)
                                 if f.lower().endswith(IMG_EXTS)])
        seq = []
        for fname in frames_sorted:
            img = cv2.imread(os.path.join(sample_dir, fname))
            if img is None:
                continue
            kp = detect_frame(img)
            if kp is not None:
                seq.append(kp)

        if len(seq) < 5:      # too few landmarks detected → skip
            failed += 1
            continue

        padded = pad_or_truncate(seq, MAX_FRAMES)   # (60, 63)
        X.append(padded)
        y.append(gloss)
        extracted += 1

    print(f'  {gloss}: {extracted}/{len(samples)} extracted')

detector.close()

X = np.array(X, dtype=np.float32)   # (N, 60, 63)
y = np.array(y)
np.savez_compressed(OUTPUT_PATH, X=X, y=y)

print(f'\n✅ Saved {len(X)} samples → {OUTPUT_PATH}')
print(f'   Failed (< 5 frames detected) : {failed}')
print(f'   Classes                      : {sorted(set(y))}')
print(f'   Shape                        : X={X.shape}  y={y.shape}')

if missing:
    print(f'\n⚠️  No folder found for: {missing}')
    print('   → Record these glosses first in Step 2.')

Downloaded: hand_landmarker.task


I0000 00:00:1778386718.166536 14887863 init-domain.cc:128] Fiber init: default domain = pthread, concurrency = 8, prefix = pthread-default
I0000 00:00:1778386718.285403 14887863 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M1
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1778386718.297247 14887865 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778386718.306901 14887870 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
GOOD (16 samples):   0%|          | 0/16 [00:00<?, ?it/s]W0000 00:00:1778386718.466066 14887872 landmark_projection_calculator.cc:81] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


  GOOD: 16/16 extracted


  MORNING: 17/17 extracted


  NIGHT: 16/16 extracted


...

  X: 5/5 extracted


Y (6 samples):  67%|██████▋   | 4/6 [00:06<00:03,  1.73s/it]E0000 00:00:1778387498.490038 14887864 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-05-10T10:04:38.428462+05:30
=== Source Location Trace: ===
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180


  Y: 6/6 extracted


E0000 00:00:1778387507.968249 14887864 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-05-10T10:04:38.428462+05:30
=== Source Location Trace: ===
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180


  Z: 4/4 extracted

✅ Saved 447 samples → data/csl_keypoints.npz
   Failed (< 5 frames detected) : 10
   Classes                      : [np.str_('A'), np.str_('AFTERNOON'), np.str_('B'), np.str_('C'), np.str_('D'), np.str_('DRINK'), np.str_('E'), np.str_('EAT'), np.str_('F'), np.str_('FINE'), np.str_('G'), np.str_('GOOD'), np.str_('H'), np.str_('HELP'), np.str_('HOME'), np.str_('HOW'), np.str_('I'), np.str_('J'), np.str_('K'), np.str_('L'), np.str_('M'), np.str_('MEET'), np.str_('MORNING'), np.str_('N'), np.str_('NAME'), np.str_('NICE'), np.str_('NIGHT'), np.str_('O'), np.str_('P'), np.str_('PLEASE'), np.str_('Q'), np.str_('R'), np.str_('S'), np.str_('SORRY'), np.str_('T'), np.str_('THANK'), np.str_('U'), np.str_('V'), np.str_('W'), np.str_('WANT'), np.str_('WATER'), np.str_('WHAT'), np.str_('WHO'), np.str_('X'), np.str_('Y'), np.str_('YOU'), np.str_('Z')]
   Shape                        : X=(447, 60, 63)  y=(447,)


In [6]:
import os
import numpy as np
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# ── Load keypoints (saved by extraction cell) ─────────────────────────────
data  = np.load('data/csl_keypoints.npz', allow_pickle=True)
X     = data['X']          # (N, 60, 63) — already wrist-normalised
y     = data['y']          # (N,) string labels

print(f'Loaded: X={X.shape}  y={y.shape}')

# ── Label encode ──────────────────────────────────────────────────────────
le        = LabelEncoder()
y_enc     = le.fit_transform(y)
NUM_CLASSES = len(le.classes_)
print(f'Classes ({NUM_CLASSES}): {list(le.classes_)}')

# ── Train / val split ─────────────────────────────────────────────────────
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc)

tr_dl  = DataLoader(TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr)),
                    batch_size=32, shuffle=True)
val_dl = DataLoader(TensorDataset(torch.tensor(X_val), torch.tensor(y_val)),
                    batch_size=32)

# ── Model ─────────────────────────────────────────────────────────────────
class GlossLSTM(nn.Module):
    def __init__(self, input_dim=63, hidden=256, layers=2,
                 num_classes=47, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden, layers,
                            batch_first=True, dropout=dropout, bidirectional=True)
        self.attn = nn.Linear(hidden * 2, 1)
        self.head = nn.Sequential(
            nn.Linear(hidden * 2, 128), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        w = torch.softmax(self.attn(out), dim=1)
        return self.head((w * out).sum(dim=1))

device  = torch.device('cpu')
model   = GlossLSTM(num_classes=NUM_CLASSES).to(device)
optim   = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
sched   = torch.optim.lr_scheduler.ReduceLROnPlateau(optim, factor=0.5, patience=3)
loss_fn = nn.CrossEntropyLoss()

print(f'\nTraining on {len(X_tr)} samples, validating on {len(X_val)}')
print(f'Model params: {sum(p.numel() for p in model.parameters()):,}\n')

# ── Training loop ─────────────────────────────────────────────────────────
EPOCHS = 40
best_acc, best_state = 0.0, None

for ep in range(1, EPOCHS + 1):
    # Train
    model.train()
    tr_loss, tr_correct = 0.0, 0
    for xb, yb in tr_dl:
        xb, yb = xb.to(device), yb.to(device)
        optim.zero_grad()
        out  = model(xb)
        loss = loss_fn(out, yb)
        loss.backward()
        optim.step()
        tr_loss    += loss.item() * len(xb)
        tr_correct += (out.argmax(1) == yb).sum().item()
    tr_loss /= len(X_tr)
    tr_acc   = tr_correct / len(X_tr)

    # Validate
    model.eval()
    val_loss, val_correct = 0.0, 0
    with torch.no_grad():
        for xb, yb in val_dl:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            val_loss    += loss_fn(out, yb).item() * len(xb)
            val_correct += (out.argmax(1) == yb).sum().item()
    val_loss /= len(X_val)
    val_acc   = val_correct / len(X_val)

    sched.step(val_loss)

    if val_acc > best_acc:
        best_acc   = val_acc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        marker = ' ← best'
    else:
        marker = ''

    print(f'Ep {ep:02d}  TrL={tr_loss:.3f} TrAcc={tr_acc*100:.1f}%  '
          f'VaL={val_loss:.3f} VaAcc={val_acc*100:.1f}%{marker}')

print(f'\n✅ Best val accuracy: {best_acc*100:.1f}%')

# ── Save ──────────────────────────────────────────────────────────────────
WORD_GLOSSES  = ['GOOD','MORNING','NIGHT','AFTERNOON','HOW','YOU','FINE','THANKS A LOT',
                 'PLEASE','HELP','SORRY','WHAT','DRINK','WATER','EAT','HOME',
                 'NICE','NAME','MEET','WHO','WANT','NAMASTE']
ALPHA_GLOSSES = [chr(i) for i in range(ord('A'), ord('Z') + 1)]

os.makedirs('csl_models', exist_ok=True)
torch.save({
    'model_state':   best_state,
    'arch': {
        'input_dim':   63,
        'hidden':      256,
        'layers':      2,
        'num_classes': NUM_CLASSES,
        'dropout':     0.3,
    },
    'label_encoder': le,
    'classes':       list(le.classes_),
    'max_frames':    60,
    'word_glosses':  WORD_GLOSSES,
    'alpha_glosses': ALPHA_GLOSSES,
    'accuracy':      best_acc,
}, 'csl_models/csl_model_lstm.pt')

print('✅ Saved → csl_models/csl_model_lstm.pt')
print('▶  Next: run csl_server.py')

Loaded: X=(447, 60, 63)  y=(447,)
Classes (47): [np.str_('A'), np.str_('AFTERNOON'), np.str_('B'), np.str_('C'), np.str_('D'), np.str_('DRINK'), np.str_('E'), np.str_('EAT'), np.str_('F'), np.str_('FINE'), np.str_('G'), np.str_('GOOD'), np.str_('H'), np.str_('HELP'), np.str_('HOME'), np.str_('HOW'), np.str_('I'), np.str_('J'), np.str_('K'), np.str_('L'), np.str_('M'), np.str_('MEET'), np.str_('MORNING'), np.str_('N'), np.str_('NAME'), np.str_('NICE'), np.str_('NIGHT'), np.str_('O'), np.str_('P'), np.str_('PLEASE'), np.str_('Q'), np.str_('R'), np.str_('S'), np.str_('SORRY'), np.str_('T'), np.str_('THANK'), np.str_('U'), np.str_('V'), np.str_('W'), np.str_('WANT'), np.str_('WATER'), np.str_('WHAT'), np.str_('WHO'), np.str_('X'), np.str_('Y'), np.str_('YOU'), np.str_('Z')]

Training on 357 samples, validating on 90
Model params: 2,306,608

Ep 01  TrL=3.754 TrAcc=6.2%  VaL=3.414 VaAcc=10.0% ← best
Ep 02  TrL=3.329 TrAcc=12.3%  VaL=3.055 VaAcc=21.1% ← best
Ep 03  TrL=3.001 TrAcc=18.2%  VaL=

In [7]:
# Test the sentence builder logic (this same code goes inside csl_server.py)

GRAMMAR_RULES = [
    # ── Greetings ──────────────────────────────────────────────────────────
    (['GOOD', 'MORNING'],          'Good morning!',              10),
    (['GOOD', 'NIGHT'],            'Good night!',                10),
    (['GOOD', 'AFTERNOON'],        'Good afternoon!',            10),
    # ── Questions ──────────────────────────────────────────────────────────
    (['HOW', 'YOU'],               'How are you?',               10),
    (['YOU', 'HOW'],               'How are you?',               10),
    (['WHAT', 'YOU', 'WANT'],      'What do you want?',          11),
    (['WHO', 'YOU'],               'Who are you?',               10),
    (['WHO'],                      'Who is this?',                7),
    # ── Name + fingerspelling  (NAME + any non-gloss word = spelled name) ──
    (['NAME'],                     'What is your name?',          8),
    # ── Responses ──────────────────────────────────────────────────────────
    (['FINE', 'THANK'],            'I am fine, thank you!',      10),
    (['THANK', 'YOU'],             'Thank you!',                 10),
    (['THANK'],                    'Thank you!',                  8),
    (['SORRY'],                    'I am sorry.',                 8),
    # ── Requests ───────────────────────────────────────────────────────────
    (['PLEASE', 'HELP'],           'Please help me.',            10),
    (['HELP', 'PLEASE'],           'Please help me.',            10),
    (['WANT', 'WATER'],            'I want water.',              10),
    (['WANT', 'DRINK'],            'I want to drink.',           10),
    (['WANT', 'EAT'],              'I want to eat.',             10),
    (['WATER', 'PLEASE'],          'Please give me water.',      10),
    (['EAT', 'PLEASE'],            'I want to eat, please.',     10),
    (['DRINK', 'PLEASE'],          'I want to drink, please.',   10),
    # ── Home / meet ────────────────────────────────────────────────────────
    (['HOME'],                     'I am going home.',            8),
    (['MEET', 'YOU'],              'Nice to meet you!',          10),
    (['NICE', 'MEET', 'YOU'],      'Nice to meet you!',          11),
    # ── Fallback single-word rules ─────────────────────────────────────────
    (['GOOD'],                     'Good.',                       6),
    (['FINE'],                     'I am fine.',                  7),
    (['HELP'],                     'I need help.',                7),
    (['WATER'],                    'I want water.',               7),
    (['EAT'],                      'I want to eat.',              7),
    (['DRINK'],                    'I want to drink.',            7),
    (['PLEASE'],                   'Please.',                     7),
    (['WANT'],                     'I want something.',           6),
]

GLOSS_MAP = {
    'GOOD':'good','MORNING':'morning','NIGHT':'night','AFTERNOON':'afternoon',
    'HOW':'how','YOU':'you','FINE':'fine','THANK':'thank you','PLEASE':'please',
    'HELP':'help','SORRY':'sorry','WHAT':'what','DRINK':'drink','WATER':'water',
    'EAT':'eat','HOME':'home','NICE':'nice','NAME':'name','MEET':'meet',
    'WHO':'who','WANT':'want',
}

ALPHA_SET = set('ABCDEFGHIJKLMNOPQRSTUVWXYZ')


def merge_fingerspelling(glosses):
    """
    Join consecutive single-letter glosses into words.
    e.g. ['NAME', 'R', 'A', 'J', 'U', 'THANK']
      -> ['NAME', 'RAJU', 'THANK']
    """
    merged, letter_buf = [], []
    for g in glosses:
        if g in ALPHA_SET:        # single letter
            letter_buf.append(g)
        else:                     # word gloss — flush any accumulated letters first
            if letter_buf:
                merged.append(''.join(letter_buf))  # e.g. 'RAJU'
                letter_buf = []
            merged.append(g)
    if letter_buf:
        merged.append(''.join(letter_buf))
    return merged


def glosses_to_sentence(raw_glosses):
    if not raw_glosses:
        return {'sentence': '', 'method': 'empty', 'confidence': 0.0}

    raw_glosses = [g.upper().strip() for g in raw_glosses]
    glosses     = merge_fingerspelling(raw_glosses)

    # ── NAME + spelled word → "My name is Xxx." ──────────────────────────
    for i, g in enumerate(glosses):
        if g == 'NAME' and i + 1 < len(glosses):
            next_g = glosses[i + 1]
            # next token is a fingerspelled word (all caps, not a known gloss)
            if next_g not in GLOSS_MAP and next_g not in ('NAME',):
                name_str = next_g.capitalize()     # 'RAJU' → 'Raju'
                return {
                    'sentence':   f'My name is {name_str}.',
                    'method':     'name_fingerspelling',
                    'confidence': 0.92,
                    'name':       name_str,
                }
        # Also handle just a fingerspelled word after WHO/MEET etc.

    # ── Check if the whole thing is just a fingerspelled word ─────────────
    if len(glosses) == 1 and glosses[0] not in GLOSS_MAP:
        word = glosses[0].capitalize()
        return {
            'sentence':   word + '.',
            'method':     'fingerspelling',
            'confidence': 0.85,
        }

    # ── Exact rule match ──────────────────────────────────────────────────
    word_glosses_only = [g for g in glosses if g not in ALPHA_SET]
    for candidate in (glosses, word_glosses_only):
        best, best_p = None, -1
        for pat, sent, pri in GRAMMAR_RULES:
            if pat == candidate and pri > best_p:
                best, best_p = sent, pri
        if best:
            return {'sentence': best, 'method': 'rule_match', 'confidence': 0.95}

    # ── Partial match ─────────────────────────────────────────────────────
    best, best_p = None, -1
    for pat, sent, pri in sorted(GRAMMAR_RULES, key=lambda x: -x[2]):
        if all(p in glosses for p in pat) and pri > best_p:
            best, best_p = sent, pri
    if best:
        return {'sentence': best, 'method': 'partial_match', 'confidence': 0.80}

    # ── Fallback: gloss map ───────────────────────────────────────────────
    words = []
    for g in glosses:
        if g in GLOSS_MAP:
            words.append(GLOSS_MAP[g])
        else:
            words.append(g.lower())    # fingerspelled word stays lower-case
    s = ' '.join(words).capitalize()
    if not s.endswith(('.', '!', '?')): s += '.'
    return {'sentence': s, 'method': 'fallback', 'confidence': 0.50}


# ── Quick tests ────────────────────────────────────────────────────────────
tests = [
    ['NAME', 'R', 'A', 'J', 'U'],             # My name is Raju.
    ['NAME', 'P', 'R', 'I', 'Y', 'A'],        # My name is Priya.
    ['GOOD', 'MORNING'],                        # Good morning!
    ['HOW', 'YOU'],                             # How are you?
    ['FINE', 'THANK'],                          # I am fine, thank you!
    ['WANT', 'WATER'],                          # I want water.
    ['PLEASE', 'HELP'],                         # Please help me.
    ['R', 'A', 'J', 'U'],                       # Raju. (just spelling)
    ['THANK', 'YOU'],                           # Thank you!
    ['NICE', 'MEET', 'YOU'],                    # Nice to meet you!
]
print('=== Sentence Builder Tests ===\n')
for t in tests:
    result = glosses_to_sentence(t)
    print(f'  Input : {t}')
    print(f'  Output: "{result["sentence"]}"  [{result["method"]}  conf={result["confidence"]}]')
    print()

=== Sentence Builder Tests ===

  Input : ['NAME', 'R', 'A', 'J', 'U']
  Output: "My name is Raju."  [name_fingerspelling  conf=0.92]

  Input : ['NAME', 'P', 'R', 'I', 'Y', 'A']
  Output: "My name is Priya."  [name_fingerspelling  conf=0.92]

  Input : ['GOOD', 'MORNING']
  Output: "Good morning!"  [rule_match  conf=0.95]

  Input : ['HOW', 'YOU']
  Output: "How are you?"  [rule_match  conf=0.95]

  Input : ['FINE', 'THANK']
  Output: "I am fine, thank you!"  [rule_match  conf=0.95]

  Input : ['WANT', 'WATER']
  Output: "I want water."  [rule_match  conf=0.95]

  Input : ['PLEASE', 'HELP']
  Output: "Please help me."  [rule_match  conf=0.95]

  Input : ['R', 'A', 'J', 'U']
  Output: "Raju."  [fingerspelling  conf=0.85]

  Input : ['THANK', 'YOU']
  Output: "Thank you!"  [rule_match  conf=0.95]

  Input : ['NICE', 'MEET', 'YOU']
  Output: "Nice to meet you!"  [rule_match  conf=0.95]

